In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import numpy as np
import torch.nn as nn
import pandas as pd

from utils.reproducibility_utils import seed_everything
from utils.data_loading import load_data, load_test_data

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [2]:
# ============================================================================
# FIXED COMPONENTS (proven optimal)
# ============================================================================

class RNN(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.rnn = nn.RNN(input_size, 64, 1, nonlinearity='relu', batch_first=True)
        self.fc = nn.Linear(64, 64)
        self.out = nn.Linear(64, 1)
        self.act = nn.ReLU()
    
    def forward(self, x):
        x, _ = self.rnn(x)
        return self.out(self.act(self.fc(x[:, -1, :])))


class WeightedL1(nn.Module):
    """FIXED: slope=1.0, threshold=2.0"""
    def forward(self, pred, target):
        weights = 1.0 + 1.0 * torch.clamp(torch.abs(target) - 2.0, min=0)
        return (torch.abs(pred - target) * weights).mean()


def predict(model, loader):
    model.eval()
    with torch.no_grad():
        return np.concatenate([model(x.to(device)).cpu().numpy() for x, _ in loader]).flatten()

print("✅ Components ready")

✅ Components ready


In [3]:
print("Loading data...")
train_loader, val_loader, pipeline = load_data(
    train_path = "../datasets_challenge/train",
    train_ratio = 0.98,
    batchsize = 64,
    columns_to_drop = ["sideSlip", "vyCG", "time", "longitude","latitude"],
    sequence_length = 20,
    threshold = True,
    threshold_value = 3.0,
    threshold_column = "vxCG",
    target = "sideSlip",
    scaling = True,
    seed = 42,
)

X_sample, y_sample = next(iter(train_loader))
input_size = X_sample.shape[2]
# Load test data
print("\nLoading test data...")
test_loader = load_test_data(
    test_path = "../datasets_challenge/test",
    batchsize = 512,
    pipeline = pipeline,
    load_anonymized_data=True
)
print("✅ Test data loaded")
print(f"✅ Data loaded")
print(f"   Input size: {input_size}")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")

Loading data...
../datasets_challenge/train/train_part_001.csv
../datasets_challenge/train/train_part_002.csv
../datasets_challenge/train/train_part_003.csv
../datasets_challenge/train/train_part_004.csv
../datasets_challenge/train/train_part_004.csv
../datasets_challenge/train/train_part_005.csv
../datasets_challenge/train/train_part_006.csv
../datasets_challenge/train/train_part_005.csv
../datasets_challenge/train/train_part_006.csv
../datasets_challenge/train/train_part_007.csv
../datasets_challenge/train/train_part_008.csv
../datasets_challenge/train/train_part_009.csv
../datasets_challenge/train/train_part_010.csv
../datasets_challenge/train/train_part_007.csv
../datasets_challenge/train/train_part_008.csv
../datasets_challenge/train/train_part_009.csv
../datasets_challenge/train/train_part_010.csv
../datasets_challenge/train/train_part_011.csv
../datasets_challenge/train/train_part_011.csv
../datasets_challenge/train/train_part_012.csv
../datasets_challenge/train/train_part_013.c

In [4]:
def train_two_stage(seed, s2_lr, s2_epochs, verbose=True):
    """
    FIXED Stage 1: L1, lr=0.0003, 30 epochs
    VARIABLE Stage 2: WeightedL1, lr=s2_lr, s2_epochs epochs
    """
    seed_everything(seed)
    model = RNN(input_size).to(device)
    
    # Stage 1: Standard L1 (FIXED)
    if verbose: print(f"  Stage 1: L1, lr=0.0003, 30ep")
    opt = torch.optim.Adam(model.parameters(), lr=0.0003)
    crit = nn.L1Loss()
    best, state = np.inf, None
    
    for ep in range(30):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(crit(model(x.to(device)), y.to(device).reshape(-1,1)).item() for x,y in val_loader) / len(val_loader)
        if val < best: best, state = val, model.state_dict().copy()
    
    model.load_state_dict(state)
    if verbose: print(f"    Best: {best:.5f}")
    
    # Stage 2: Weighted L1 (VARIABLE lr and epochs)
    if verbose: print(f"  Stage 2: WeightedL1, lr={s2_lr}, {s2_epochs}ep")
    opt = torch.optim.Adam(model.parameters(), lr=s2_lr)
    crit2 = WeightedL1()
    best2, state2 = np.inf, None
    
    for ep in range(s2_epochs):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit2(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(nn.L1Loss()(model(x.to(device)), y.to(device).reshape(-1,1)).item() for x,y in val_loader) / len(val_loader)
        if val < best2: best2, state2 = val, model.state_dict().copy()
    
    model.load_state_dict(state2)
    if verbose: print(f"    Best: {best2:.5f}")
    
    return model, best2


def train_baseline(seed, verbose=True):
    """Standard L1 only (FIXED)"""
    seed_everything(seed)
    model = RNN(input_size).to(device)
    if verbose: print(f"  Baseline: L1, lr=0.0003, early stop")
    
    opt = torch.optim.Adam(model.parameters(), lr=0.0003)
    crit = nn.L1Loss()
    best, state, patience = np.inf, None, 0
    
    for ep in range(50):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(crit(model(x.to(device)), y.to(device).reshape(-1,1)).item() for x,y in val_loader) / len(val_loader)
        if val < best: 
            best, state, patience = val, model.state_dict().copy(), 0
        else:
            patience += 1
            if patience > 10: break
    
    model.load_state_dict(state)
    if verbose: print(f"    Best: {best:.5f}")
    return model, best

print("✅ Training functions ready")

✅ Training functions ready


## 🎯 Run Focused Experiments

In [5]:
# FIXED: seeds 42 and 2024 (proven best)
SEEDS = [42, 2024]

# FOCUSED: Only vary Stage 2 LR and epochs (continuing winning trend)
CONFIGS = [
    # (name, s2_lr, s2_epochs) - all with fixed slope=1.0, thresh=2.0
    ("lr3e5_40ep", 0.00003, 40),   # Lower LR, more epochs
    ("lr5e5_40ep", 0.00005, 40),   # Winning LR, more epochs
    ("lr2e5_50ep", 0.00002, 50),   # Even lower LR, even more epochs
    ("lr3e5_35ep", 0.00003, 35),   # Balance point
]

print("="*60)
print("🎯 FOCUSED EXPERIMENTS: LR & Epochs only")
print("   Fixed: slope=1.0, threshold=2.0, seeds=[42,2024], blend=50/50")
print("="*60)

🎯 FOCUSED EXPERIMENTS: LR & Epochs only
   Fixed: slope=1.0, threshold=2.0, seeds=[42,2024], blend=50/50


In [6]:
# Train baseline models (FIXED - needed for 50/50 blend)
print("\n📦 BASELINE MODELS")
print("-"*40)

baseline_preds = []
for seed in SEEDS:
    print(f"\n🌱 Seed {seed}")
    model, _ = train_baseline(seed)
    baseline_preds.append(predict(model, test_loader))

baseline_ensemble = np.mean(baseline_preds, axis=0)
print(f"\n✅ Baseline ensemble ready")


📦 BASELINE MODELS
----------------------------------------

🌱 Seed 42
  Baseline: L1, lr=0.0003, early stop
    Best: 0.11366
    Best: 0.11366

🌱 Seed 2024
  Baseline: L1, lr=0.0003, early stop

🌱 Seed 2024
  Baseline: L1, lr=0.0003, early stop
    Best: 0.11157
    Best: 0.11157

✅ Baseline ensemble ready

✅ Baseline ensemble ready


In [7]:
# Train two-stage models with different configs
results = {}

for name, s2_lr, s2_epochs in CONFIGS:
    print(f"\n{'='*60}")
    print(f"📦 CONFIG: {name} (lr={s2_lr}, epochs={s2_epochs})")
    print("="*60)
    
    config_preds = []
    for seed in SEEDS:
        print(f"\n🌱 Seed {seed}")
        model, val_loss = train_two_stage(seed, s2_lr, s2_epochs)
        config_preds.append(predict(model, test_loader))
    
    results[name] = np.mean(config_preds, axis=0)
    print(f"\n✅ {name} done")


📦 CONFIG: lr3e5_40ep (lr=3e-05, epochs=40)

🌱 Seed 42
  Stage 1: L1, lr=0.0003, 30ep
    Best: 0.12387
  Stage 2: WeightedL1, lr=3e-05, 40ep
    Best: 0.12387
  Stage 2: WeightedL1, lr=3e-05, 40ep
    Best: 0.11275
    Best: 0.11275

🌱 Seed 2024
  Stage 1: L1, lr=0.0003, 30ep

🌱 Seed 2024
  Stage 1: L1, lr=0.0003, 30ep
    Best: 0.12751
  Stage 2: WeightedL1, lr=3e-05, 40ep
    Best: 0.12751
  Stage 2: WeightedL1, lr=3e-05, 40ep
    Best: 0.11361
    Best: 0.11361

✅ lr3e5_40ep done

📦 CONFIG: lr5e5_40ep (lr=5e-05, epochs=40)

🌱 Seed 42
  Stage 1: L1, lr=0.0003, 30ep

✅ lr3e5_40ep done

📦 CONFIG: lr5e5_40ep (lr=5e-05, epochs=40)

🌱 Seed 42
  Stage 1: L1, lr=0.0003, 30ep
    Best: 0.12588
  Stage 2: WeightedL1, lr=5e-05, 40ep
    Best: 0.12588
  Stage 2: WeightedL1, lr=5e-05, 40ep
    Best: 0.11190
    Best: 0.11190

🌱 Seed 2024
  Stage 1: L1, lr=0.0003, 30ep

🌱 Seed 2024
  Stage 1: L1, lr=0.0003, 30ep
    Best: 0.12434
  Stage 2: WeightedL1, lr=5e-05, 40ep
    Best: 0.12434
  Stage 2:

In [8]:
# Save predictions (50/50 blend with baseline)
print("\n" + "="*60)
print("💾 SAVING PREDICTIONS (50/50 blend)")
print("="*60)

for name, twostage_ens in results.items():
    # 50/50 blend (FIXED optimal ratio)
    blend = (baseline_ensemble + twostage_ens) / 2
    filename = f"pred_v7_{name}.csv"
    pd.DataFrame({"id": range(len(blend)), "predicted_sideslip": blend}).to_csv(filename, index=False)
    print(f"✅ {filename}")

print("\n" + "="*60)
print("📋 SUBMIT TO KAGGLE:")
print("="*60)
print("Current best: 0.21957 (lr=0.00005, 30ep)")
print("")
for name, s2_lr, s2_epochs in CONFIGS:
    print(f"  - pred_v7_{name}.csv  (lr={s2_lr}, {s2_epochs}ep)")
print("")
print("💡 Hypothesis: Lower LR + More epochs → better few-shot")


💾 SAVING PREDICTIONS (50/50 blend)
✅ pred_v7_lr3e5_40ep.csv
✅ pred_v7_lr5e5_40ep.csv
✅ pred_v7_lr2e5_50ep.csv
✅ pred_v7_lr3e5_35ep.csv

📋 SUBMIT TO KAGGLE:
Current best: 0.21957 (lr=0.00005, 30ep)

  - pred_v7_lr3e5_40ep.csv  (lr=3e-05, 40ep)
  - pred_v7_lr5e5_40ep.csv  (lr=5e-05, 40ep)
  - pred_v7_lr2e5_50ep.csv  (lr=2e-05, 50ep)
  - pred_v7_lr3e5_35ep.csv  (lr=3e-05, 35ep)

💡 Hypothesis: Lower LR + More epochs → better few-shot
✅ pred_v7_lr3e5_35ep.csv

📋 SUBMIT TO KAGGLE:
Current best: 0.21957 (lr=0.00005, 30ep)

  - pred_v7_lr3e5_40ep.csv  (lr=3e-05, 40ep)
  - pred_v7_lr5e5_40ep.csv  (lr=5e-05, 40ep)
  - pred_v7_lr2e5_50ep.csv  (lr=2e-05, 50ep)
  - pred_v7_lr3e5_35ep.csv  (lr=3e-05, 35ep)

💡 Hypothesis: Lower LR + More epochs → better few-shot
